# GROW Stream-Chemistry Campaign — Environmental Context

The [GROW](https://growobservatory.org/) project collects stream-water chemistry samples at field sites worldwide.  
This notebook retrieves multi-source environmental context for five representative sampling events — two from the  
Yukon River (Alaska, 2004) and three from the Pacific Northwest and mid-Atlantic (2019) — using a ±7-day window  
around each collection date.

| Source | Variables | Coverage |
|---|---|---|
| NASA POWER MERRA-2 | Temperature, precipitation, humidity, wind, soil moisture | 1980–present · 0.5° |
| NASA POWER SYN1deg | SW ↓, PAR, LW ↓ (all-sky & clear-sky) | 2001–present · 1° |
| SSURGO | Dominant soil map unit | US sites only |
| SoilGrids v2 | Bulk density, clay, pH, sand, silt, organic carbon | Global · 0–5 cm |
| GBIF | Species occurrences within 50 km (±180 days) | Global |
| OpenAQ | Surface air quality | Requires `OPENAQ_API_KEY` |
| Sentinel-5P TROPOMI | CO total column | Yakima River site · collection date |


In [ ]:
from datetime import datetime, timedelta

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from env_data_mcp.sources.gbif import gbif_occurrences
from env_data_mcp.sources.nasa_power import (
    TemporalResolution,
    nasa_power_merra2_query,
    nasa_power_syn1deg_query,
)
from env_data_mcp.sources.openaq import openaq_query
from env_data_mcp.sources.sentinel5p import sentinel5p_query
from env_data_mcp.sources.soilgrids import soilgrids_query
from env_data_mcp.sources.ssurgo import ssurgo_query

GROW_SAMPLES = [
    {"name": "Yukon (Apr 2004)", "date": "2004-04-07", "lat": 61.93333333, "lon": -162.8666667},
    {"name": "Yukon (Jun 2004)", "date": "2004-06-15", "lat": 61.93333333, "lon": -162.8666667},
    {"name": "Yakima River", "date": "2019-08-19", "lat": 46.2531882, "lon": -119.4768203},
    {"name": "White Clay Ck. 1", "date": "2019-08-12", "lat": 39.8577967, "lon": -75.7830393},
    {"name": "White Clay Ck. 2", "date": "2019-08-12", "lat": 39.8594333, "lon": -75.7839486},
]

SITE_COLORS = ["#1f77b4", "#aec7e8", "#2ca02c", "#d62728", "#ff7f0e"]


def date_window(date: str, days: int = 7) -> tuple[str, str]:
    d = datetime.strptime(date, "%Y-%m-%d")
    return (
        (d - timedelta(days=days)).strftime("%Y-%m-%d"),
        (d + timedelta(days=days)).strftime("%Y-%m-%d"),
    )

## 1. Fetch Environmental Context Data

All sources are queried below. Data are collected into plain dicts/DataFrames and visualised in §2.

### NASA POWER — MERRA-2 meteorology and SYN1deg surface radiation


In [ ]:
merra2_dfs: dict[str, pd.DataFrame] = {}
syn1deg_dfs: dict[str, pd.DataFrame] = {}

for s in GROW_SAMPLES:
    start, end = date_window(s["date"])

    r_m = nasa_power_merra2_query(
        latitude=s["lat"],
        longitude=s["lon"],
        start_date=start,
        end_date=end,
        temporal_resolution=TemporalResolution.DAILY,
    )
    if r_m["data"]:
        df = pd.DataFrame(r_m["data"])
        df["date"] = pd.to_datetime(df["date"])
        merra2_dfs[s["name"]] = df

    r_s = nasa_power_syn1deg_query(
        latitude=s["lat"],
        longitude=s["lon"],
        start_date=start,
        end_date=end,
        temporal_resolution=TemporalResolution.DAILY,
    )
    if r_s["data"]:
        df = pd.DataFrame(r_s["data"])
        df["date"] = pd.to_datetime(df["date"])
        syn1deg_dfs[s["name"]] = df

### Soil Properties — SSURGO and SoilGrids v2


In [ ]:
ssurgo_results: dict = {}
for s in GROW_SAMPLES:
    ssurgo_results[s["name"]] = ssurgo_query(latitude=s["lat"], longitude=s["lon"])

SoilGrids returns point estimates at 250 m resolution for the top 5 cm of soil (bulk density, clay, pH, sand, silt, organic carbon).


In [ ]:
soilgrids_results: dict = {}
for s in GROW_SAMPLES:
    soilgrids_results[s["name"]] = soilgrids_query(latitude=s["lat"], longitude=s["lon"])

### Biodiversity — GBIF species occurrences


In [ ]:
gbif_results: dict = {}
for s in GROW_SAMPLES:
    start, end = date_window(s["date"], days=180)
    gbif_results[s["name"]] = gbif_occurrences(
        latitude=s["lat"],
        longitude=s["lon"],
        radius_km=50.0,
        start_date=start,
        end_date=end,
        limit=200,
    )

### Air Quality — OpenAQ

OpenAQ v3 requires a free API key (`OPENAQ_API_KEY` env var). Responses with no key return `auth_present=False`; the loop below handles both cases.


In [ ]:
openaq_results: dict = {}
for s in GROW_SAMPLES:
    start, end = date_window(s["date"])
    openaq_results[s["name"]] = openaq_query(
        latitude=s["lat"],
        longitude=s["lon"],
        radius_km=100.0,
        start_date=start,
        end_date=end,
        limit=100,
    )

### Atmospheric CO — Sentinel-5P TROPOMI

One overpass is queried for the Yakima River site on the collection date. Each S5P granule is ~150 MB; expect 30–120 s.


In [ ]:
yakima = next(s for s in GROW_SAMPLES if "Yakima" in s["name"])
s5p_yakima = sentinel5p_query(
    latitude=yakima["lat"],
    longitude=yakima["lon"],
    start_date=yakima["date"],
    end_date=yakima["date"],
    product="CO",
)

## 2. Analysis

### Site Map


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for s, color in zip(GROW_SAMPLES, SITE_COLORS, strict=False):
    ax.scatter(
        s["lon"],
        s["lat"],
        color=color,
        s=120,
        zorder=5,
        edgecolors="white",
        linewidth=0.8,
        label=s["name"],
    )
    ax.annotate(
        s["name"],
        xy=(s["lon"], s["lat"]),
        xytext=(6, 4),
        textcoords="offset points",
        fontsize=8,
        color=color,
    )

ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
ax.set_title("GROW Stream-Chemistry Sampling Locations")
ax.legend(fontsize=8, framealpha=0.9, loc="lower right")
ax.grid(True, alpha=0.3, linewidth=0.5)
for sp in ax.spines.values():
    sp.set_linewidth(0.5)
plt.tight_layout()
plt.show()

### Hydrometeorological & Radiation Context at Collection Time

Three panels per site (column):  
- **Temperature** — daily mean with min–max shaded band (MERRA-2)  
- **Precipitation** — daily total (MERRA-2)  
- **PAR** — all-sky (solid) vs clear-sky (dashed); shaded area is the cloud reduction (SYN1deg)  

The dashed vertical line marks the collection date.


In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(18, 10), sharex="col")

for col, (s, color) in enumerate(zip(GROW_SAMPLES, SITE_COLORS, strict=False)):
    name = s["name"]
    sample_date = pd.to_datetime(s["date"])
    ax_t, ax_p, ax_par = axes[0, col], axes[1, col], axes[2, col]

    # --- Temperature (MERRA-2) ---
    if name in merra2_dfs:
        df = merra2_dfs[name]
        ax_t.fill_between(df["date"], df["T2M_MIN"], df["T2M_MAX"], alpha=0.2, color=color)
        ax_t.plot(df["date"], df["T2M"], color=color, linewidth=1.5)
    ax_t.axvline(sample_date, color="k", linestyle="--", linewidth=0.8, alpha=0.6)
    ax_t.set_title(f"{name}\n{s['date']}", fontsize=8)

    # --- Precipitation (MERRA-2) ---
    if name in merra2_dfs:
        df = merra2_dfs[name]
        ax_p.bar(df["date"], df["PRECTOTCORR"], color=color, alpha=0.7, width=0.9)
    ax_p.axvline(sample_date, color="k", linestyle="--", linewidth=0.8, alpha=0.6)

    # --- PAR (SYN1deg) ---
    if name in syn1deg_dfs:
        df = syn1deg_dfs[name]
        ax_par.fill_between(
            df["date"], df["ALLSKY_SFC_PAR_TOT"], df["CLRSKY_SFC_PAR_TOT"], alpha=0.25, color=color
        )
        ax_par.plot(
            df["date"],
            df["CLRSKY_SFC_PAR_TOT"],
            color=color,
            linestyle="--",
            linewidth=0.9,
            alpha=0.6,
        )
        ax_par.plot(df["date"], df["ALLSKY_SFC_PAR_TOT"], color=color, linewidth=1.5)
    ax_par.axvline(sample_date, color="k", linestyle="--", linewidth=0.8, alpha=0.6)

    for ax in (ax_t, ax_p, ax_par):
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=4))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=35, ha="right", fontsize=7)
        ax.tick_params(axis="y", labelsize=7)
        ax.yaxis.set_major_locator(plt.MaxNLocator(4))
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

axes[0, 0].set_ylabel("Temperature (°C)", fontsize=9)
axes[1, 0].set_ylabel("Precipitation\n(mm day⁻¹)", fontsize=9)
axes[2, 0].set_ylabel("PAR\n(W m⁻²)", fontsize=9)

legend_elements = [
    Line2D([0], [0], color="k", linestyle="--", linewidth=1, label="Collection date"),
    Line2D([0], [0], color="gray", linewidth=1.5, label="Mean (T) / All-sky (PAR)"),
    Patch(facecolor="gray", alpha=0.2, label="T daily range / Cloud reduction in PAR"),
]
fig.legend(handles=legend_elements, loc="upper right", fontsize=8, framealpha=0.9, ncol=3)
fig.suptitle(
    "Hydrometeorological & Radiation Context at GROW Sampling Sites (±7 days)\n"
    "MERRA-2 meteorology  ·  SYN1deg surface radiation",
    fontsize=11,
    y=1.01,
)
plt.tight_layout()
plt.show()

### Top-Soil Properties (SoilGrids v2, 0–5 cm)


In [ ]:
SOIL_PROPS = [
    ("soc", "Soil Organic\nCarbon (g/kg)"),
    ("clay", "Clay (%)"),
    ("phh2o", "pH (H₂O)"),
    ("bdod", "Bulk Density\n(kg dm⁻³)"),
]

short_names = [s["name"].replace("White Clay Ck.", "WCC") for s in GROW_SAMPLES]
x = np.arange(len(GROW_SAMPLES))

fig, axes = plt.subplots(1, len(SOIL_PROPS), figsize=(14, 4))

for ax, (prop, label) in zip(axes, SOIL_PROPS, strict=False):
    for xi, (s, color) in enumerate(zip(GROW_SAMPLES, SITE_COLORS, strict=False)):
        val = soilgrids_results.get(s["name"], {}).get("data", {}).get(prop)
        if val is not None:
            bar = ax.bar(xi, val, color=color, alpha=0.85, edgecolor="white", linewidth=0.5)
            ax.text(
                xi,
                val + ax.get_ylim()[1] * 0.01,
                f"{val:.1f}",
                ha="center",
                va="bottom",
                fontsize=7,
            )
    ax.set_xticks(x)
    ax.set_xticklabels(short_names, rotation=30, ha="right", fontsize=8)
    ax.set_title(label, fontsize=9)
    ax.grid(axis="y", alpha=0.3, linewidth=0.5)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle("SoilGrids v2 — Top-Soil Properties at GROW Sites (0–5 cm depth)", fontsize=11)
plt.tight_layout()
plt.show()

### Species Richness — GBIF occurrences within 50 km (±180 days)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(GROW_SAMPLES))

# Compute counts first so we can set a sensible ylim before drawing annotations.
counts = []
for s in GROW_SAMPLES:
    records = gbif_results.get(s["name"], {}).get("data", [])
    counts.append(len({r.get("species") for r in records if r.get("species")}))

y_top = max(max(counts) * 1.25, 5)  # always at least 5 so the chart is visible
ax.set_ylim(0, y_top)

for xi, (_s, color, n) in enumerate(zip(GROW_SAMPLES, SITE_COLORS, counts, strict=False)):
    ax.bar(xi, n, color=color, alpha=0.85, edgecolor="white", linewidth=0.5)
    ax.text(xi, n + y_top * 0.02, str(n), ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([s["name"] for s in GROW_SAMPLES], rotation=15, ha="right")
ax.set_ylabel("Unique species observed")
ax.set_title("GBIF Species Richness within 50 km  (±180 days of collection date)")
ax.grid(axis="y", alpha=0.3, linewidth=0.5)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.subplots_adjust(bottom=0.2, top=0.91)
plt.show()